# DiCo-NLI Track 1 — TF-IDF + Logistic Regression

Deneyler EXP-01–EXP-05 ve grup bazlı doğrulama. Hücreleri sırayla çalıştırın. Veriler ve resmi değerlendirme deposu ilk hücrelerde indirilir.


In [ ]:
!git clone https://github.com/betulcelik1/NLP_Project.git

In [ ]:
!git clone https://github.com/ilopezgazpio/SemEval-2027-Task-2-DiCo-NLI.git

In [ ]:
!ls /content

In [ ]:
!find /content/SemEval-2027-Task-2-DiCo-NLI -type f | grep participant_labeled

In [ ]:
import pandas as pd

In [ ]:
train_path = "/content/SemEval-2027-Task-2-DiCo-NLI/final_data/train/dico_nli_train_track1_participant_labeled.csv"

dev_path = "/content/SemEval-2027-Task-2-DiCo-NLI/final_data/dev/dico_nli_dev_track1_participant_labeled.csv"

train_df = pd.read_csv(train_path)
dev_df = pd.read_csv(dev_path)

print("Train:", train_df.shape)
print("Dev:", dev_df.shape)

In [ ]:
print("Train columns:")
print(train_df.columns.tolist())

print("\nDev columns:")
print(dev_df.columns.tolist())

In [ ]:
display(train_df.head())

In [ ]:
print(train_df.info())

print("\nMissing values:")
print(train_df.isnull().sum())

In [ ]:
print("Train label distribution:")
print(train_df["label"].value_counts())

print("\nTrain label distribution (percentage):")
print((train_df["label"].value_counts(normalize=True) * 100).round(2))

print("\nDev label distribution:")
print(dev_df["label"].value_counts())

In [ ]:
for i in range(5):
    print(f"\n--- Example {i+1} ---")
    print("Text 1:", train_df.loc[i, "text1"])
    print("Text 2:", train_df.loc[i, "text2"])
    print("Label:", train_df.loc[i, "label"])

## EXP-01 — Baseline ve hata analizi


In [ ]:
train_texts = (
    "Text1: " + train_df["text1"].astype(str) +
    " Text2: " + train_df["text2"].astype(str)
)

dev_texts = (
    "Text1: " + dev_df["text1"].astype(str) +
    " Text2: " + dev_df["text2"].astype(str)
)

y_train = train_df["label"]
y_dev = dev_df["label"]

print(train_texts.iloc[0])
print("\nLabel:", y_train.iloc[0])

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50000
)

X_train = tfidf.fit_transform(train_texts)
X_dev = tfidf.transform(dev_texts)

print("Train feature matrix:", X_train.shape)
print("Dev feature matrix:", X_dev.shape)
print("Vocabulary size:", len(tfidf.vocabulary_))

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

lr_model.fit(X_train, y_train)

print("Model training completed.")

In [ ]:
y_pred = lr_model.predict(X_dev)

print("First 10 predictions:")
print(y_pred[:10])

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

accuracy = accuracy_score(y_dev, y_pred)
macro_f1 = f1_score(y_dev, y_pred, average="macro")
weighted_f1 = f1_score(y_dev, y_pred, average="weighted")

print("Accuracy:", round(accuracy, 4))
print("Macro F1:", round(macro_f1, 4))
print("Weighted F1:", round(weighted_f1, 4))

print("\nClassification Report:")
print(classification_report(y_dev, y_pred))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

labels = [
    "EQUIVALENCE",
    "FORWARD_ENTAILMENT",
    "BACKWARD_ENTAILMENT",
    "NEGATIVE_OTHER"
]

cm = confusion_matrix(y_dev, y_pred, labels=labels)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels
)

fig, ax = plt.subplots(figsize=(10, 8))
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45)
plt.title("TF-IDF + Logistic Regression - Dev Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

cm_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

print(cm_df)

In [ ]:
error_df = dev_df.copy()

error_df["predicted_label"] = y_pred
error_df["correct"] = error_df["label"] == error_df["predicted_label"]

wrong_predictions = error_df[error_df["correct"] == False].copy()

print("Total incorrect predictions:", len(wrong_predictions))
print("Total correct predictions:", (error_df["correct"]).sum())

display(
    wrong_predictions[
        ["text1", "text2", "label", "predicted_label"]
    ].head(20)
)

In [ ]:
wrong_predictions["error_type"] = (
    wrong_predictions["label"] + " -> " +
    wrong_predictions["predicted_label"]
)

print(wrong_predictions["error_type"].value_counts())

In [ ]:
import json
import os

results_dir = "/content/NLP_Project/results/baseline"
os.makedirs(results_dir, exist_ok=True)

baseline_metrics = {
    "model": "TF-IDF + Logistic Regression",
    "accuracy": float(accuracy),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "train_size": int(len(train_df)),
    "dev_size": int(len(dev_df)),
    "vocabulary_size": int(len(tfidf.vocabulary_)),
    "ngram_range": [1, 2],
    "max_features": 50000,
    "class_weight": "balanced"
}

with open(
    os.path.join(results_dir, "metrics.json"),
    "w"
) as f:
    json.dump(baseline_metrics, f, indent=4)

print("Metrics saved.")

In [ ]:
predictions_df = pd.DataFrame({
    "instance_id": dev_df["instance_id"],
    "pair_id": dev_df["pair_id"],
    "text1": dev_df["text1"],
    "text2": dev_df["text2"],
    "true_label": y_dev,
    "predicted_label": y_pred
})

predictions_df.to_csv(
    os.path.join(results_dir, "dev_predictions.csv"),
    index=False
)

print("Predictions saved.")

In [ ]:
pair_analysis = dev_df.copy()

pair_analysis["predicted_label"] = y_pred

pair_sizes = pair_analysis.groupby("pair_id").size()

print("Pair size distribution:")
print(pair_sizes.value_counts())

In [ ]:
paired_ids = pair_sizes[pair_sizes == 2].index

paired_df = pair_analysis[
    pair_analysis["pair_id"].isin(paired_ids)
].sort_values("pair_id")

display(
    paired_df[
        ["pair_id", "text1", "text2", "label", "predicted_label"]
    ].head(20)
)

In [ ]:
pair_eval = dev_df.copy()

pair_eval["predicted_label"] = y_pred
pair_eval["is_correct"] = pair_eval["label"] == pair_eval["predicted_label"]

paired_eval = pair_eval.groupby("pair_id").filter(
    lambda group: len(group) == 2
)

pair_results = []

for pair_id, group in paired_eval.groupby("pair_id"):
    correct_count = group["is_correct"].sum()

    if correct_count == 2:
        status = "Both correct"
    elif correct_count == 1:
        status = "One correct"
    else:
        status = "Both incorrect"

    pair_results.append({
        "pair_id": pair_id,
        "status": status,
        "correct_count": correct_count
    })

pair_results_df = pd.DataFrame(pair_results)

print(pair_results_df["status"].value_counts())

print("\nPair-level accuracy:")
print(
    round(
        pair_results_df["correct_count"].sum() /
        (len(pair_results_df) * 2),
        4
    )
)

In [ ]:
import matplotlib.pyplot as plt

pair_status_counts = pair_results_df["status"].value_counts()

plt.figure(figsize=(8, 5))
pair_status_counts.plot(kind="bar")
plt.title("Paired Samples: Prediction Outcomes")
plt.ylabel("Number of pairs")
plt.xlabel("Pair outcome")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
error_analysis_path = os.path.join(
    results_dir,
    "error_analysis.csv"
)

wrong_predictions.to_csv(
    error_analysis_path,
    index=False
)

print("Error analysis saved:", error_analysis_path)

## Resmi Dev değerlendirmesi


In [ ]:
import os
import pandas as pd

submission_path = os.path.join(
    results_dir,
    "dev_submission.csv"
)

submission_df = pd.DataFrame({
    "instance_id": dev_df["instance_id"],
    "label": y_pred
})

submission_df.to_csv(submission_path, index=False)

print("Submission shape:", submission_df.shape)
print("Columns:", submission_df.columns.tolist())
print("Duplicate IDs:", submission_df["instance_id"].duplicated().sum())

display(submission_df.head())

In [ ]:
import subprocess

official_repo = "/content/SemEval-2027-Task-2-DiCo-NLI"

reference_path = os.path.join(
    official_repo,
    "final_data/dev/dico_nli_dev_track1_reference.csv"
)

score_dir = os.path.join(results_dir, "official_scores")

command = [
    "python", "-m", "evaluation_functions",
    "--gold", reference_path,
    "--predictions", submission_path,
    "--output-dir", score_dir
]

result = subprocess.run(
    command,
    cwd=official_repo,
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)

In [ ]:
import json

if result.returncode == 0:
    with open(os.path.join(score_dir, "scores.json")) as f:
        official_scores = json.load(f)

    print("Official scorer output:")
    print(json.dumps(official_scores, indent=2))
else:
    print("Scoring failed. Check the previous cell.")

In [ ]:
print("Train:", train_df.shape)
print("Dev:", dev_df.shape)

print("Train columns:", train_df.columns.tolist())

print("Model:", type(model).__name__ if "model" in globals() else "Not found")
print("Vectorizer:", type(vectorizer).__name__ if "vectorizer" in globals() else "Not found")

In [ ]:
print("Baseline model:", type(lr_model).__name__)
print("Baseline vectorizer:", type(tfidf).__name__)

print("Train:", train_df.shape)
print("Dev:", dev_df.shape)

## EXP-02 — Ayrı kelime TF-IDF


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

# text1 ve text2 için ayrı vektörleştiriciler
tfidf_text1 = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50000
)

tfidf_text2 = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50000
)

# Eğitim verisi
X_train_1 = tfidf_text1.fit_transform(train_df["text1"])
X_train_2 = tfidf_text2.fit_transform(train_df["text2"])

# Doğrulama verisi
X_dev_1 = tfidf_text1.transform(dev_df["text1"])
X_dev_2 = tfidf_text2.transform(dev_df["text2"])

# Özellikleri birleştir
X_train_v2 = hstack([X_train_1, X_train_2]).tocsr()
X_dev_v2 = hstack([X_dev_1, X_dev_2]).tocsr()

print("Train feature matrix:", X_train_v2.shape)
print("Dev feature matrix:", X_dev_v2.shape)

print("Text1 vocabulary:", len(tfidf_text1.vocabulary_))
print("Text2 vocabulary:", len(tfidf_text2.vocabulary_))

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model_v2 = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

lr_model_v2.fit(X_train_v2, y_train)

print("Iteration 3 model training completed.")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

y_pred_v2 = lr_model_v2.predict(X_dev_v2)

accuracy_v2 = accuracy_score(y_dev, y_pred_v2)
macro_f1_v2 = f1_score(y_dev, y_pred_v2, average="macro")
weighted_f1_v2 = f1_score(y_dev, y_pred_v2, average="weighted")

print("Iteration 3 Results")
print("Accuracy:", round(accuracy_v2, 4))
print("Macro F1:", round(macro_f1_v2, 4))
print("Weighted F1:", round(weighted_f1_v2, 4))

print("\nBaseline vs Iteration 3")
print("Baseline Weighted F1:", round(weighted_f1, 4))
print("New Weighted F1:", round(weighted_f1_v2, 4))
print("Difference:", round(weighted_f1_v2 - weighted_f1, 4))

In [ ]:
results_dir_v2 = "/content/NLP_Project/results/iteration_3"
os.makedirs(results_dir_v2, exist_ok=True)

submission_v2 = pd.DataFrame({
    "instance_id": dev_df["instance_id"],
    "label": y_pred_v2
})

submission_path_v2 = os.path.join(
    results_dir_v2, "dev_submission.csv"
)

submission_v2.to_csv(submission_path_v2, index=False)

print("Shape:", submission_v2.shape)
print("Duplicate IDs:", submission_v2["instance_id"].duplicated().sum())

In [ ]:
import subprocess
import json

score_dir_v2 = os.path.join(results_dir_v2, "official_scores")

command_v2 = [
    "python", "-m", "evaluation_functions",
    "--gold", reference_path,
    "--predictions", submission_path_v2,
    "--output-dir", score_dir_v2
]

result_v2 = subprocess.run(
    command_v2,
    cwd=official_repo,
    capture_output=True,
    text=True
)

print("Return code:", result_v2.returncode)

if result_v2.returncode == 0:
    with open(os.path.join(score_dir_v2, "scores.json")) as f:
        scores_v2 = json.load(f)

    print("Weighted F1:", round(scores_v2["weighted_f1"], 4))
    print("SoftCons:", round(scores_v2["soft_cons"], 4))
    print("HardCons:", round(scores_v2["hard_cons"], 4))
else:
    print(result_v2.stderr)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

labels = [
    "BACKWARD_ENTAILMENT",
    "EQUIVALENCE",
    "FORWARD_ENTAILMENT",
    "NEGATIVE_OTHER"
]

print(classification_report(
    y_dev,
    y_pred_v2,
    labels=labels,
    digits=4
))

cm_v2 = confusion_matrix(
    y_dev,
    y_pred_v2,
    labels=labels
)

display(pd.DataFrame(
    cm_v2,
    index=labels,
    columns=labels
))

In [ ]:
error_df_v2 = dev_df.copy()
error_df_v2["predicted_label"] = y_pred_v2

negative_errors = error_df_v2[
    (error_df_v2["label"] == "NEGATIVE_OTHER") &
    (error_df_v2["predicted_label"] != "NEGATIVE_OTHER")
]

print("Total NEGATIVE_OTHER errors:", len(negative_errors))

display(
    negative_errors[
        ["text1", "text2", "label", "predicted_label"]
    ].head(10)
)

## EXP-03 — Sözcüksel özellikler


In [ ]:
import numpy as np
from scipy.sparse import hstack, csr_matrix

def extract_extra_features(df):
    features = []

    for _, row in df.iterrows():
        words1 = set(str(row["text1"]).lower().split())
        words2 = set(str(row["text2"]).lower().split())

        intersection = len(words1 & words2)
        union = len(words1 | words2)

        jaccard = intersection / union if union else 0

        overlap1 = intersection / len(words1) if words1 else 0
        overlap2 = intersection / len(words2) if words2 else 0

        length_diff = abs(len(words1) - len(words2))
        length_diff_norm = length_diff / max(
            len(words1), len(words2), 1
        )

        features.append([
            jaccard,
            overlap1,
            overlap2,
            length_diff_norm
        ])

    return np.array(features, dtype=float)

extra_train = extract_extra_features(train_df)
extra_dev = extract_extra_features(dev_df)

X_train_v3 = hstack([
    X_train_v2,
    csr_matrix(extra_train)
]).tocsr()

X_dev_v3 = hstack([
    X_dev_v2,
    csr_matrix(extra_dev)
]).tocsr()

print("Extra train features:", extra_train.shape)
print("Extra dev features:", extra_dev.shape)
print("New train matrix:", X_train_v3.shape)
print("New dev matrix:", X_dev_v3.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model_v3 = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

lr_model_v3.fit(X_train_v3, train_df["label"])

print("EXP-03 training completed.")
print("Model:", type(lr_model_v3).__name__)
print("Features:", lr_model_v3.n_features_in_)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

y_pred_v3 = lr_model_v3.predict(X_dev_v3)

results_comparison = pd.DataFrame({
    "Model": ["EXP-01", "EXP-02", "EXP-03"],
    "Accuracy": [
        0.4500,
        accuracy_score(y_dev, y_pred_v2),
        accuracy_score(y_dev, y_pred_v3)
    ],
    "Macro F1": [
        0.4449,
        f1_score(y_dev, y_pred_v2, average="macro"),
        f1_score(y_dev, y_pred_v3, average="macro")
    ],
    "Weighted F1": [
        0.4612,
        f1_score(y_dev, y_pred_v2, average="weighted"),
        f1_score(y_dev, y_pred_v3, average="weighted")
    ]
})

display(results_comparison.round(4))

print(
    "EXP-03 vs EXP-02 Weighted F1 difference:",
    round(
        results_comparison.loc[2, "Weighted F1"]
        - results_comparison.loc[1, "Weighted F1"],
        4
    )
)

In [ ]:
import os
import json
import subprocess
import pandas as pd

results_dir_v3 = "/content/NLP_Project/results/exp_03"
os.makedirs(results_dir_v3, exist_ok=True)

submission_v3 = pd.DataFrame({
    "instance_id": dev_df["instance_id"],
    "label": y_pred_v3
})

submission_path_v3 = os.path.join(
    results_dir_v3, "dev_submission.csv"
)

submission_v3.to_csv(submission_path_v3, index=False)

score_dir_v3 = os.path.join(results_dir_v3, "official_scores")

command_v3 = [
    "python", "-m", "evaluation_functions",
    "--gold", reference_path,
    "--predictions", submission_path_v3,
    "--output-dir", score_dir_v3
]

result_v3 = subprocess.run(
    command_v3,
    cwd=official_repo,
    capture_output=True,
    text=True
)

print("Submission shape:", submission_v3.shape)
print("Duplicate IDs:", submission_v3["instance_id"].duplicated().sum())
print("Return code:", result_v3.returncode)

if result_v3.returncode == 0:
    with open(os.path.join(score_dir_v3, "scores.json")) as f:
        scores_v3 = json.load(f)

    print("\nEXP-03 Official Scores")
    print("Weighted F1:", round(scores_v3["weighted_f1"], 4))
    print("SoftCons:", round(scores_v3["soft_cons"], 4))
    print("HardCons:", round(scores_v3["hard_cons"], 4))
else:
    print(result_v3.stdout)
    print(result_v3.stderr)

In [ ]:
from sklearn.metrics import classification_report

print("=== EXP-03 CLASSIFICATION REPORT ===")
print(classification_report(y_dev, y_pred_v3, digits=4))

print("\n=== NEGATIVE_OTHER COMPARISON ===")

for name, predictions in [
    ("EXP-02", y_pred_v2),
    ("EXP-03", y_pred_v3)
]:
    mask = dev_df["label"] == "NEGATIVE_OTHER"

    correct = (
        predictions[mask] == dev_df.loc[mask, "label"]
    ).sum()

    print(f"{name}: {correct}/{mask.sum()} correct")

## Ablation ve EXP-04 — Jaccard olmadan


In [ ]:
from scipy.sparse import hstack, csr_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
import pandas as pd
import numpy as np

feature_names = [
    "Jaccard",
    "Overlap1",
    "Overlap2",
    "LengthDiff"
]

ablation_results = []

for i, removed_feature in enumerate(feature_names):

    keep_indices = [j for j in range(4) if j != i]

    X_train_ablation = hstack([
        X_train_v2,
        csr_matrix(extra_train[:, keep_indices])
    ]).tocsr()

    X_dev_ablation = hstack([
        X_dev_v2,
        csr_matrix(extra_dev[:, keep_indices])
    ]).tocsr()

    model = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    )

    model.fit(X_train_ablation, y_train)
    predictions = model.predict(X_dev_ablation)

    ablation_results.append({
        "Removed Feature": removed_feature,
        "Accuracy": accuracy_score(y_dev, predictions),
        "Weighted F1": f1_score(
            y_dev, predictions, average="weighted"
        )
    })

ablation_df = pd.DataFrame(ablation_results)

ablation_df["F1 Difference vs EXP-03"] = (
    ablation_df["Weighted F1"]
    - f1_score(y_dev, y_pred_v3, average="weighted")
)

display(ablation_df.round(4))

In [ ]:
from scipy.sparse import hstack, csr_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# Jaccard (indeks 0) çıkarılıyor
selected_indices = [1, 2, 3]

X_train_v4 = hstack([
    X_train_v2,
    csr_matrix(extra_train[:, selected_indices])
]).tocsr()

X_dev_v4 = hstack([
    X_dev_v2,
    csr_matrix(extra_dev[:, selected_indices])
]).tocsr()

lr_model_v4 = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

lr_model_v4.fit(X_train_v4, y_train)
y_pred_v4 = lr_model_v4.predict(X_dev_v4)

print("EXP-04 feature count:", X_train_v4.shape[1])
print("Accuracy:", round(accuracy_score(y_dev, y_pred_v4), 4))
print("Macro F1:", round(f1_score(y_dev, y_pred_v4, average="macro"), 4))
print("Weighted F1:", round(f1_score(y_dev, y_pred_v4, average="weighted"), 4))

In [ ]:
import os
import json
import subprocess
import pandas as pd

results_dir_v4 = "/content/NLP_Project/results/exp_04"
os.makedirs(results_dir_v4, exist_ok=True)

submission_v4 = pd.DataFrame({
    "instance_id": dev_df["instance_id"],
    "label": y_pred_v4
})

submission_path_v4 = os.path.join(
    results_dir_v4, "dev_submission.csv"
)

submission_v4.to_csv(submission_path_v4, index=False)

score_dir_v4 = os.path.join(
    results_dir_v4, "official_scores"
)

result_v4 = subprocess.run(
    [
        "python", "-m", "evaluation_functions",
        "--gold", reference_path,
        "--predictions", submission_path_v4,
        "--output-dir", score_dir_v4
    ],
    cwd=official_repo,
    capture_output=True,
    text=True
)

print("Submission shape:", submission_v4.shape)
print("Duplicate IDs:", submission_v4["instance_id"].duplicated().sum())
print("Return code:", result_v4.returncode)

if result_v4.returncode == 0:
    with open(os.path.join(score_dir_v4, "scores.json")) as f:
        scores_v4 = json.load(f)

    print("\nEXP-04 Official Scores")
    print("Weighted F1:", round(scores_v4["weighted_f1"], 4))
    print("SoftCons:", round(scores_v4["soft_cons"], 4))
    print("HardCons:", round(scores_v4["hard_cons"], 4))
else:
    print(result_v4.stdout)
    print(result_v4.stderr)

In [ ]:
comparison_df = dev_df[
    ["instance_id", "text1", "text2", "label"]
].copy()

comparison_df["EXP-03"] = y_pred_v3
comparison_df["EXP-04"] = y_pred_v4

changed = comparison_df[
    comparison_df["EXP-03"] != comparison_df["EXP-04"]
].copy()

changed["EXP-03 Correct"] = (
    changed["EXP-03"] == changed["label"]
)

changed["EXP-04 Correct"] = (
    changed["EXP-04"] == changed["label"]
)

print("Changed predictions:", len(changed))
print("EXP-04 fixed:", (
    ~changed["EXP-03 Correct"] &
    changed["EXP-04 Correct"]
).sum())

print("EXP-04 worsened:", (
    changed["EXP-03 Correct"] &
    ~changed["EXP-04 Correct"]
).sum())

display(changed.head(15))

## EXP-05 — Karakter n-gramları


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

char_vectorizer_1 = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    max_features=10000,
    min_df=2
)

char_vectorizer_2 = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    max_features=10000,
    min_df=2
)

X_train_char1 = char_vectorizer_1.fit_transform(
    train_df["text1"].fillna("")
)
X_dev_char1 = char_vectorizer_1.transform(
    dev_df["text1"].fillna("")
)

X_train_char2 = char_vectorizer_2.fit_transform(
    train_df["text2"].fillna("")
)
X_dev_char2 = char_vectorizer_2.transform(
    dev_df["text2"].fillna("")
)

X_train_v5 = hstack([
    X_train_v4,
    X_train_char1,
    X_train_char2
]).tocsr()

X_dev_v5 = hstack([
    X_dev_v4,
    X_dev_char1,
    X_dev_char2
]).tocsr()

print("Train shape:", X_train_v5.shape)
print("Dev shape:", X_dev_v5.shape)
print("Character features:", X_train_v5.shape[1] - X_train_v4.shape[1])

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model_v5 = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

lr_model_v5.fit(X_train_v5, y_train)

print("EXP-05 training completed.")
print("Model:", type(lr_model_v5).__name__)
print("Feature count:", lr_model_v5.n_features_in_)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)
import pandas as pd

y_pred_v5 = lr_model_v5.predict(X_dev_v5)

comparison_v5 = pd.DataFrame({
    "Model": ["EXP-04", "EXP-05"],
    "Accuracy": [
        accuracy_score(y_dev, y_pred_v4),
        accuracy_score(y_dev, y_pred_v5)
    ],
    "Macro F1": [
        f1_score(y_dev, y_pred_v4, average="macro"),
        f1_score(y_dev, y_pred_v5, average="macro")
    ],
    "Weighted F1": [
        f1_score(y_dev, y_pred_v4, average="weighted"),
        f1_score(y_dev, y_pred_v5, average="weighted")
    ]
})

display(comparison_v5.round(4))

print("\nEXP-05 vs EXP-04 Weighted F1 difference:")
print(round(
    comparison_v5.loc[1, "Weighted F1"]
    - comparison_v5.loc[0, "Weighted F1"],
    4
))

print("\n=== EXP-05 Classification Report ===")
print(classification_report(y_dev, y_pred_v5, digits=4))

In [ ]:
import os
import json
import subprocess
import pandas as pd

results_dir_v5 = "/content/NLP_Project/results/exp_05"
os.makedirs(results_dir_v5, exist_ok=True)

submission_v5 = pd.DataFrame({
    "instance_id": dev_df["instance_id"],
    "label": y_pred_v5
})

submission_path_v5 = os.path.join(
    results_dir_v5, "dev_submission.csv"
)

submission_v5.to_csv(submission_path_v5, index=False)

score_dir_v5 = os.path.join(
    results_dir_v5, "official_scores"
)

result_v5 = subprocess.run(
    [
        "python", "-m", "evaluation_functions",
        "--gold", reference_path,
        "--predictions", submission_path_v5,
        "--output-dir", score_dir_v5
    ],
    cwd=official_repo,
    capture_output=True,
    text=True
)

print("Submission shape:", submission_v5.shape)
print("Duplicate IDs:", submission_v5["instance_id"].duplicated().sum())
print("Return code:", result_v5.returncode)

if result_v5.returncode == 0:
    with open(os.path.join(score_dir_v5, "scores.json")) as f:
        scores_v5 = json.load(f)

    print("\nEXP-05 Official Scores")
    print("Weighted F1:", round(scores_v5["weighted_f1"], 4))
    print("SoftCons:", round(scores_v5["soft_cons"], 4))
    print("HardCons:", round(scores_v5["hard_cons"], 4))
else:
    print(result_v5.stdout)
    print(result_v5.stderr)

## İç doğrulama — pair_id bazlı


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        train_df,
        groups=train_df["pair_id"]
    )
)

train_internal = train_df.iloc[train_idx].copy()
val_internal = train_df.iloc[val_idx].copy()

train_pairs = set(train_internal["pair_id"])
val_pairs = set(val_internal["pair_id"])

print("Internal train:", train_internal.shape)
print("Internal validation:", val_internal.shape)
print(
    "Overlapping pair IDs:",
    len(train_pairs & val_pairs)
)

print("\nTraining label distribution:")
print(train_internal["label"].value_counts())

print("\nValidation label distribution:")
print(val_internal["label"].value_counts())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

# EXP-05: Ayrı kelime ve karakter TF-IDF
word_1 = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50000
)

word_2 = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50000
)

char_1 = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    max_features=10000,
    min_df=2
)

char_2 = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    max_features=10000,
    min_df=2
)

train_parts = []
val_parts = []

for vectorizer, column in [
    (word_1, "text1"),
    (word_2, "text2"),
    (char_1, "text1"),
    (char_2, "text2")
]:
    train_parts.append(
        vectorizer.fit_transform(
            train_internal[column].fillna("")
        )
    )

    val_parts.append(
        vectorizer.transform(
            val_internal[column].fillna("")
        )
    )

# EXP-04'te koruduğumuz üç ek özellik
extra_train_internal = extract_extra_features(train_internal)[:, 1:4]
extra_val_internal = extract_extra_features(val_internal)[:, 1:4]

X_train_internal = hstack(
    train_parts + [csr_matrix(extra_train_internal)]
).tocsr()

X_val_internal = hstack(
    val_parts + [csr_matrix(extra_val_internal)]
).tocsr()

print("Internal train matrix:", X_train_internal.shape)
print("Internal validation matrix:", X_val_internal.shape)
print("Feature dimensions match:",
      X_train_internal.shape[1] == X_val_internal.shape[1])

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

internal_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

internal_model.fit(
    X_train_internal,
    train_internal["label"]
)

internal_pred = internal_model.predict(X_val_internal)

print("=== GROUP-BASED VALIDATION ===")

print(
    "Accuracy:",
    round(accuracy_score(val_internal["label"], internal_pred), 4)
)

print(
    "Macro F1:",
    round(f1_score(
        val_internal["label"],
        internal_pred,
        average="macro"
    ), 4)
)

print(
    "Weighted F1:",
    round(f1_score(
        val_internal["label"],
        internal_pred,
        average="weighted"
    ), 4)
)

print("\nClassification Report:")
print(classification_report(
    val_internal["label"],
    internal_pred,
    digits=4
))

## Tekrar kontrolü ve temiz grup ayrımı


In [ ]:
def normalize_text(s):
    return " ".join(str(s).lower().split())

def make_pair_key(df):
    return df.apply(
        lambda row: tuple(sorted([
            normalize_text(row["text1"]),
            normalize_text(row["text2"])
        ])),
        axis=1
    )

train_keys = set(make_pair_key(train_internal))
val_keys = set(make_pair_key(val_internal))

overlap = train_keys & val_keys

print("Overlapping pair IDs:",
      len(set(train_internal["pair_id"]) &
          set(val_internal["pair_id"])))

print("Overlapping normalized text pairs:", len(overlap))

if overlap:
    print("\nExample overlaps:")
    for pair in list(overlap)[:10]:
        print(pair)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd

# Aynı pair_id veya aynı metin çiftini paylaşan
# örnekleri tek bir grupta birleştiriyoruz.

parent = list(range(len(train_df)))

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

seen_pair_ids = {}
seen_text_pairs = {}

text_keys = make_pair_key(train_df).tolist()

for i, (pair_id, text_key) in enumerate(
    zip(train_df["pair_id"], text_keys)
):
    if pair_id in seen_pair_ids:
        union(i, seen_pair_ids[pair_id])
    else:
        seen_pair_ids[pair_id] = i

    if text_key in seen_text_pairs:
        union(i, seen_text_pairs[text_key])
    else:
        seen_text_pairs[text_key] = i

groups_clean = [find(i) for i in range(len(train_df))]

splitter_clean = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx_clean, val_idx_clean = next(
    splitter_clean.split(train_df, groups=groups_clean)
)

train_clean = train_df.iloc[train_idx_clean].copy()
val_clean = train_df.iloc[val_idx_clean].copy()

pair_overlap = (
    set(train_clean["pair_id"]) &
    set(val_clean["pair_id"])
)

text_overlap = (
    set(make_pair_key(train_clean)) &
    set(make_pair_key(val_clean))
)

print("Clean train:", train_clean.shape)
print("Clean validation:", val_clean.shape)
print("Overlapping pair IDs:", len(pair_overlap))
print("Overlapping normalized text pairs:", len(text_overlap))

print("\nValidation label distribution:")
print(val_clean["label"].value_counts())

## EXP-05 — Temiz grup bazlı doğrulama


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

clean_train_parts = []
clean_val_parts = []
clean_vectorizers = {}

for name, column, analyzer, ngrams, max_features, min_df in [
    ("word1", "text1", "word", (1, 2), 50000, 1),
    ("word2", "text2", "word", (1, 2), 50000, 1),
    ("char1", "text1", "char", (2, 5), 10000, 2),
    ("char2", "text2", "char", (2, 5), 10000, 2)
]:
    vec = TfidfVectorizer(
        analyzer=analyzer,
        ngram_range=ngrams,
        max_features=max_features,
        min_df=min_df
    )

    clean_train_parts.append(
        vec.fit_transform(train_clean[column].fillna(""))
    )

    clean_val_parts.append(
        vec.transform(val_clean[column].fillna(""))
    )

    clean_vectorizers[name] = vec

# EXP-05'te kullanılan üç lexical özellik
extra_train_clean = extract_extra_features(train_clean)[:, 1:4]
extra_val_clean = extract_extra_features(val_clean)[:, 1:4]

X_train_clean = hstack(
    clean_train_parts + [csr_matrix(extra_train_clean)]
).tocsr()

X_val_clean = hstack(
    clean_val_parts + [csr_matrix(extra_val_clean)]
).tocsr()

print("Clean train matrix:", X_train_clean.shape)
print("Clean validation matrix:", X_val_clean.shape)
print("Feature dimensions match:",
      X_train_clean.shape[1] == X_val_clean.shape[1])

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

clean_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

clean_model.fit(
    X_train_clean,
    train_clean["label"]
)

clean_pred = clean_model.predict(X_val_clean)
clean_gold = val_clean["label"]

print("=== EXP-05 CLEAN VALIDATION ===")

print("Accuracy:",
      round(accuracy_score(clean_gold, clean_pred), 4))

print("Macro F1:",
      round(f1_score(clean_gold, clean_pred,
                     average="macro"), 4))

print("Weighted F1:",
      round(f1_score(clean_gold, clean_pred,
                     average="weighted"), 4))

print("\nClassification Report:")
print(classification_report(
    clean_gold,
    clean_pred,
    digits=4
))